#**Large Language Models Lab(MCSE642P)**

**Ex-8 - LLMs performance evaluation**

SHRIHARIHARAN S

24MCS1058

In [1]:
!pip install transformers datasets evaluate bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [41]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import evaluate
import torch
import numpy as np
from sklearn.metrics import f1_score

In [42]:
# Setup device
device = 0 if torch.cuda.is_available() else -1

In [43]:
# Load model and tokenizer
model_name = "distilbert-base-uncased-finetuned-sst-2-english"  # fine-tuned on SST-2
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [44]:
# Inference pipeline
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=device)

Device set to use cuda:0


In [45]:
# Load SST-2 dataset
sst2 = load_dataset("glue", "sst2")
test_samples = sst2["validation"].select(np.random.randint(0, len(sst2["validation"]), size=100)) # Changed line

In [46]:
test_samples[10]

{'sentence': '( a ) shapeless blob of desperate entertainment . ',
 'label': 0,
 'idx': 681}

In [47]:
# Get predictions
pred_texts = [ex["sentence"] for ex in test_samples]

In [48]:
pred_texts

['jason x is positively anti-darwinian : nine sequels and 400 years later , the teens are none the wiser and jason still kills on auto-pilot . ',
 'the film tries too hard to be funny and tries too hard to be hip . ',
 'if you enjoy more thoughtful comedies with interesting conflicted characters ; this one is for you . ',
 "it 's a buggy drag . ",
 "it 's inoffensive , cheerful , built to inspire the young people , set to an unending soundtrack of beach party pop numbers and aside from its remarkable camerawork and awesome scenery , it 's about as exciting as a sunburn . ",
 'falls neatly into the category of good stupid fun . ',
 "the longer the movie goes , the worse it gets , but it 's actually pretty good in the first few minutes . ",
 "charles ' entertaining film chronicles seinfeld 's return to stand-up comedy after the wrap of his legendary sitcom , alongside wannabe comic adams ' attempts to get his shot at the big time . ",
 'the action switches between past and present , but 

In [49]:
pred_results = classifier(pred_texts)
pred_labels = [p["label"] if isinstance(p["label"], int) else (1 if p["label"].upper() == "POSITIVE" else 0) for p in pred_results]
true_labels = test_samples["label"] #actual Label

In [50]:
# Evaluation - GLUE Metric
glue_metric = evaluate.load("glue", "sst2")
glue_result = glue_metric.compute(predictions=pred_labels, references=true_labels)
print("GLUE Evaluation (SST-2):", glue_result)

GLUE Evaluation (SST-2): {'accuracy': 0.85}


In [51]:
# Evaluation - F1 Score
f1 = f1_score(true_labels, pred_labels)
print("F1 Score:", f1)

F1 Score: 0.8421052631578947


In [53]:
print("Predicted:", pred_labels[:20])
print("Actual:   ", true_labels[:20])
print("F1 Score: ", f1_score(true_labels, pred_labels))

Predicted: [1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0]
Actual:    [0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0]
F1 Score:  0.8421052631578947


GLUE Benchmark (SST-2)

Accuracy: 0.85 → Good sentiment classification.

F1 Score: 0.8421 → Balanced precision and recall.

In [54]:
!pip install rouge_score

In [55]:
references = [s for s in pred_texts]  # original inputs as dummy refs
generated = [s + " [Pred: " + str(label) + "]" for s, label in zip(pred_texts, pred_labels)] # sample Generation


In [56]:
references

['jason x is positively anti-darwinian : nine sequels and 400 years later , the teens are none the wiser and jason still kills on auto-pilot . ',
 'the film tries too hard to be funny and tries too hard to be hip . ',
 'if you enjoy more thoughtful comedies with interesting conflicted characters ; this one is for you . ',
 "it 's a buggy drag . ",
 "it 's inoffensive , cheerful , built to inspire the young people , set to an unending soundtrack of beach party pop numbers and aside from its remarkable camerawork and awesome scenery , it 's about as exciting as a sunburn . ",
 'falls neatly into the category of good stupid fun . ',
 "the longer the movie goes , the worse it gets , but it 's actually pretty good in the first few minutes . ",
 "charles ' entertaining film chronicles seinfeld 's return to stand-up comedy after the wrap of his legendary sitcom , alongside wannabe comic adams ' attempts to get his shot at the big time . ",
 'the action switches between past and present , but 

In [57]:
generated

['jason x is positively anti-darwinian : nine sequels and 400 years later , the teens are none the wiser and jason still kills on auto-pilot .  [Pred: 1]',
 'the film tries too hard to be funny and tries too hard to be hip .  [Pred: 0]',
 'if you enjoy more thoughtful comedies with interesting conflicted characters ; this one is for you .  [Pred: 1]',
 "it 's a buggy drag .  [Pred: 0]",
 "it 's inoffensive , cheerful , built to inspire the young people , set to an unending soundtrack of beach party pop numbers and aside from its remarkable camerawork and awesome scenery , it 's about as exciting as a sunburn .  [Pred: 0]",
 'falls neatly into the category of good stupid fun .  [Pred: 1]',
 "the longer the movie goes , the worse it gets , but it 's actually pretty good in the first few minutes .  [Pred: 1]",
 "charles ' entertaining film chronicles seinfeld 's return to stand-up comedy after the wrap of his legendary sitcom , alongside wannabe comic adams ' attempts to get his shot at t

In [58]:
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

In [59]:
bleu_score = bleu.compute(predictions=generated, references=references)
print("🟦 BLEU Score:", bleu_score)

rouge_score = rouge.compute(predictions=generated, references=references)
print("🟥 ROUGE Scores:", rouge_score)

bert_result = bertscore.compute(predictions=generated, references=references, lang="en")
print("🧠 BERTScore F1:", np.mean(bert_result["f1"]))

🟦 BLEU Score: {'bleu': 0.7934027893140796, 'precisions': [0.8059006211180124, 0.7980613893376414, 0.7895622895622896, 0.7803163444639719], 'brevity_penalty': 1.0, 'length_ratio': 1.2408477842003853, 'translation_length': 2576, 'reference_length': 2076}
🟥 ROUGE Scores: {'rouge1': np.float64(0.9346646477070406), 'rouge2': np.float64(0.9277123535049012), 'rougeL': np.float64(0.9344267030620487), 'rougeLsum': np.float64(0.9344410941974413)}


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🧠 BERTScore F1: 0.9670642459392548


**BLEU Score**

BLEU: 0.7934 → High n-gram overlap between predictions and references.

Precisions: [0.81, 0.80, 0.79, 0.78] → Consistent across n-grams (1–4).

Brevity Penalty: 1.0 → No penalty, generated text length is appropriate.

Length Ratio: 1.24 → Slightly longer than reference (may indicate more detail).

**ROUGE Scores**

ROUGE-1: 0.9347 → Excellent word-level overlap.

ROUGE-2: 0.9277 → Strong phrase-level (bigram) similarity.

ROUGE-L: 0.9344 → Excellent sequence structure alignment.

**BERTScore**

F1: 0.9670 → Outstanding semantic similarity with references.

#Evaluation Metrics for Text Generation (Method 2)

In [60]:
from evaluate import load

In [62]:
predictions = [
    "The cat is on the mat.",
    "A man is playing a guitar."
]
references = [
    ["There is a cat on the mat."],
    ["A person plays the guitar."]
]

In [63]:
bleu = load("bleu")
rouge = load("rouge")
meteor = load("meteor")
bertscore = load("bertscore")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [64]:
bleu_score = bleu.compute(predictions=predictions, references=references)
rouge_score = rouge.compute(predictions=predictions, references=[r[0] for r in references])
meteor_score = meteor.compute(predictions=predictions, references=[r[0] for r in references])
bertscore_result = bertscore.compute(predictions=predictions, references=[r[0] for r in references], lang="en")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [65]:
print("BLEU:", bleu_score['bleu'])
print("ROUGE:", rouge_score)
print("METEOR:", meteor_score['meteor'])
print("BERTScore (F1):", sum(bertscore_result['f1']) / len(bertscore_result['f1']))

BLEU: 0.2705411345269698
ROUGE: {'rouge1': np.float64(0.5664335664335663), 'rouge2': np.float64(0.1818181818181818), 'rougeL': np.float64(0.4895104895104895), 'rougeLsum': np.float64(0.4895104895104895)}
METEOR: 0.6147216746212907
BERTScore (F1): 0.9565489292144775


# **Inefernce**

**Metric	Score	Interpretation**

* BLEU	0.27 -Decent n-gram overlap — moderate lexical similarity, could be better. Often lower for creative or paraphrased output.
* ROUGE-1	0.566	- Good unigram recall — many keywords from reference are present in prediction.
* ROUGE-2	0.182	- Low bigram overlap — suggests your generated text structure differs from reference.
* ROUGE-L	0.489	Moderate longest common subsequence — preserves sentence structure to some extent.
* METEOR	0.615	 Very good — accounts for synonyms and stems. This means the generated text is semantically close to the reference.
* BERTScore (F1)	0.957	 Excellent semantic similarity! — the generated output likely captures the meaning of the reference almost perfectly, even if wording differs.

#Semantic Similarity

In [66]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')
ref = "Life is about learning every day"
gen = "Life is a journey of learning and growth"

emb1 = model.encode(ref, convert_to_tensor=True)
emb2 = model.encode(gen, convert_to_tensor=True)

similarity = util.pytorch_cos_sim(emb1, emb2)
print("Semantic Similarity:", similarity.item())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Semantic Similarity: 0.7222955822944641


semantic similarity score of 0.72 means the generated quote is quite close in meaning to the reference quote

0.9 – 1.0	Nearly identical meaning

0.7 – 0.9	Strong similarity

0.4 – 0.7	Moderately similar

0.1 – 0.4	Weak similarity

< 0.1	No meaningful similarity